In [51]:
import openai
import os

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client


### Download all data from qdrant

In [52]:
qdrant_client = QdrantClient(url="http://localhost:6333")


In [5]:
# Download all data from Qdrant

all_points = qdrant_client.scroll(
    collection_name="products",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)


In [53]:
all_points[0][0].payload

{'description': 'Sensationnel Dashly lace front synthetic wigs have a wide hand tied swiss lace parting area with HD transparent lace and baby hair. Dashly wigs are prestyled yet customizable and easy to use.',
 'image': 'https://m.media-amazon.com/images/I/71APnrecqGL._SL1500_.jpg',
 'rating_number': 782,
 'price': 28.98,
 'average_rating': 4.0,
 'parent_asin': 'B07ZHP2WJX'}

In [54]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]]


In [55]:
all_context


[{'id': 'B07ZHP2WJX',
  'text': 'Sensationnel Dashly lace front synthetic wigs have a wide hand tied swiss lace parting area with HD transparent lace and baby hair. Dashly wigs are prestyled yet customizable and easy to use.'},
 {'id': 'B08XXQFF1Y',
  'text': 'Non Magnetic Eyeliner and Eyelashes Kit, Magic Self Adhesive False Eyelashes and Eyeliner, 5 Pairs 5D Reusable False Lashes with No Glue, Waterproof Lash Boxes with Mirror & Tweezers'},
 {'id': 'B00S1IADKK',
  'text': "If you're in need of a midday refresher, spray Invictus on yourself to liven up your day. This men's fragrance reveals a number of notes, including hints of grapefruit, Hedione jasmine, patchouli, bay leaves, and oak moss. Launched by Paco Rabanne in 2013, this delightful fragrance is perfect for the man who wants to always be at his best. This flexible scent can be worn at the office or at an afternoon gathering of friends or family."},
 {'id': 'B01K8QC0PI',
  'text': 'How to Use Apply nail polish on your nails an

### Render a prompt for generating synthetic eval reference dataset

In [56]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
             "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },  
        },
    },
}


In [57]:
import json

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{all_context}
"""


In [58]:
from groq import Groq
import os

groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [70]:
from dotenv import load_dotenv
import os

# Force it to overwrite the old key in memory
load_dotenv("../.env", override=True)

# Now it should show your newly updated key
print("Key starts with:", os.environ.get("OPENAI_API_KEY"))


openai = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
)



Key starts with: sk-proj-REDACTED


In [71]:
os.environ.get("OPENAI_API_KEY"),


('sk-proj-REDACTED',)

In [ ]:
response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ]
    
)


print(response.choices[0].message.content)


```json
[
  {
    "reasoning": "This question pertains to the features of wearable accessories, for which the wig chunks provide information.",
    "question": "What features do Dashly wigs offer?",
    "chunk_ids": ["B07ZHP2WJX"],
    "answer_example": "Dashly wigs have a wide hand-tied Swiss lace parting area, HD transparent lace, and customizable baby hair. They are prestyled and easy to use.",
  },
  {
    "reasoning": "The question about false eyelashes can be answered with the information provided about the Non Magnetic Eyeliner and Eyelashes Kit.",
    "question": "How do the reusable false eyelashes work?",
    "chunk_ids": ["B08XXQFF1Y"],
    "answer_example": "The kit includes magic self-adhesive false eyelashes that require no glue, making them easy to apply and remove.",
  },
  {
    "reasoning": "Information about caring for nail polish from one chunk can answer this question.",
    "question": "How do I apply nail polish properly?",
    "chunk_ids": ["B01K8QC0PI"],
    "a

In [77]:

import json
import re

# 1. Get the raw string response
raw_output = response.choices[0].message.content

# 2. Strip out the ```json and ``` markdown markers if they exist
match = re.search(r"```(?:json)?(.*?)```", raw_output, re.DOTALL)
if match:
    raw_output = match.group(1).strip()

# 3. Use Regex to remove any invalid trailing commas before a } or ]
raw_output = re.sub(r',\s*}', '}', raw_output)
raw_output = re.sub(r',\s*]', ']', raw_output)

# 4. Now safely parse it!
json_output = json.loads(raw_output)

print(f"Successfully parsed {len(json_output)} items!")



Successfully parsed 21 items!


In [78]:
points = qdrant_client.scroll(
    collection_name="products",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B09NQFXL4K")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]


In [79]:
points[0].payload


{'description': 'Product Parameter： Weight：95g Color : Black and Brown Universally designed, easy open and close jar Long-wearing, rich, saturated black shade Glides along the lash line with no tugging, dragging, or pulling Waterproof & transfer resistant Ophthalmologist and dermatologist tested HOW TO USE : Glide the flexible tip applicator across the formula to pick up just the right amount of product for a smooth, even application of color. PLACE: Place your index finger on the flat notch of the Wand and the other side of the Wand against your cheekbone. TILT & TRACE: Tilt the Tip so it falls where the eyelashes meet the eyelid. Peek through the ‘window’ and let the Wand guide your hand as you draw across the lashline. Package Include: 1 Set x Eyeliner (inculde black cream and brown cream)',
 'image': 'https://m.media-amazon.com/images/I/517FZF9IfjL._SL1500_.jpg',
 'rating_number': 1,
 'price': 0.0,
 'average_rating': 1.0,
 'parent_asin': 'B09NQFXL4K'}

In [80]:
def get_description(parent_asin: str) -> str:

    points = qdrant_client.scroll(
        collection_name="products",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload["description"]


In [82]:
get_description("B09NQFXL4K")


'Product Parameter： Weight：95g Color : Black and Brown Universally designed, easy open and close jar Long-wearing, rich, saturated black shade Glides along the lash line with no tugging, dragging, or pulling Waterproof & transfer resistant Ophthalmologist and dermatologist tested HOW TO USE : Glide the flexible tip applicator across the formula to pick up just the right amount of product for a smooth, even application of color. PLACE: Place your index finger on the flat notch of the Wand and the other side of the Wand against your cheekbone. TILT & TRACE: Tilt the Tip so it falls where the eyelashes meet the eyelid. Peek through the ‘window’ and let the Wand guide your hand as you draw across the lashline. Package Include: 1 Set x Eyeliner (inculde black cream and brown cream)'

In [83]:
import os
from dotenv import load_dotenv

# Load the environment variables from the root project folder
load_dotenv("../.env")

# Now os.environ will contain the key!
client = Client(api_key=os.environ.get("LANGSMITH_API_KEY"))


In [84]:
dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for evaluating RAG pipeline"
)


LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

In [85]:
for item in json_output:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )
